In [1]:
import os

folder_name = 'Semantic'

# Crea la cartella se non esiste
os.makedirs(folder_name, exist_ok=True)

print(f"Cartella '{folder_name}' creata o già esistente!")

Cartella 'Semantic' creata o già esistente!


Install + unzip dei due modelli

In [2]:
!pip -q install -U transformers datasets evaluate rouge_score sacrebleu bert-score scikit-learn sentencepiece accelerate py7zr

import os, shutil, zipfile, tarfile, pathlib
import py7zr

CLF_ARCHIVE = "/content/Semantic/model_classifier_en.zip"
RWR_ARCHIVE = "/content/Semantic/model_inclusive_rewriter_en.zip"

CLF_DIR = "/content/models/classifier_en"
RWR_DIR = "/content/models/rewriter_en"

def _flatten_if_single_folder(out_dir):
    items = [p for p in pathlib.Path(out_dir).iterdir()]
    if len(items) == 1 and items[0].is_dir():
        inner = items[0]
        # sposta tutto dentro out_dir
        for p in inner.iterdir():
            shutil.move(str(p), out_dir)
        shutil.rmtree(inner)
        print("Flattened single inner folder:", inner.name)

def extract_any(archive_path, out_dir):
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)

    # prova zip
    try:
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(out_dir)
        print("Extracted as ZIP:", archive_path)
        _flatten_if_single_folder(out_dir)
        print("Files:", os.listdir(out_dir)[:20], "...")
        return
    except zipfile.BadZipFile:
        pass

    # prova tar/tar.gz/tgz
    try:
        if tarfile.is_tarfile(archive_path):
            with tarfile.open(archive_path, "r:*") as t:
                t.extractall(out_dir)
            print("Extracted as TAR:", archive_path)
            _flatten_if_single_folder(out_dir)
            print("Files:", os.listdir(out_dir)[:20], "...")
            return
    except Exception:
        pass

    # prova 7z
    try:
        with py7zr.SevenZipFile(archive_path, mode="r") as z:
            z.extractall(path=out_dir)
        print("Extracted as 7Z:", archive_path)
        _flatten_if_single_folder(out_dir)
        print("Files:", os.listdir(out_dir)[:20], "...")
        return
    except Exception as e:
        raise RuntimeError(
            f"Non riesco a estrarre {archive_path}. "
            f"Probabile formato non supportato o file corrotto/incompleto. Errore: {e}"
        )

print("== Extract classifier ==")
extract_any(CLF_ARCHIVE, CLF_DIR)

print("\n== Extract rewriter ==")
extract_any(RWR_ARCHIVE, RWR_DIR)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.6 MB/s eta 0:00:00
   ━━━

Caricamento dataset EN + split

In [ ]:
from datasets import load_dataset, DatasetDict
import random

SEED = 42
random.seed(SEED)

# Riscrittore (coppie non_inclusiva -> inclusiva)
REWRITER_PATH = "/content/Semantic/dataset_en.json"

# Classificatore (già splittato)
CLF_TRAIN_PATH = "/content/Semantic/dataset_002_en/train_classification.json"
CLF_TEST_PATH  = "/content/Semantic/dataset_002_en/test_classification.json"


# Classificatore
TEXT_COL = "text"
LABEL_COL = "label_id"   # oppure "label" / "labels"

# Riscrittore
SRC_COL = "non_inclusiva"
TGT_COL = "inclusiva"

# ==============================
# LOAD CLASSIFICATORE (train/test)
# ==============================
clf_splits = load_dataset("json", data_files={
    "train": CLF_TRAIN_PATH,
    "test":  CLF_TEST_PATH
})

print("=== CLASSIFIER DATASET ===")
print(clf_splits)
print("CLF columns:", clf_splits["train"].column_names)
print("CLF row0:", clf_splits["train"][0])

# ==============================
rew_ds = load_dataset("json", data_files={"data": REWRITER_PATH})["data"]

print("\n=== REWRITER DATASET ===")
print("Rows:", len(rew_ds))
print("Cols:", rew_ds.column_names)
print("Row0:", rew_ds[0])

# Split solo per evaluation (90/10)
tmp = rew_ds.train_test_split(test_size=0.1, seed=SEED)
rew_splits = DatasetDict({
    "train": tmp["train"],       # se ti serve per sanity/preview
    "test": tmp["test"],         # usiamo questo per metriche riscrittore
})

print("\nREWRITER SPLITS:", rew_splits)

# ==============================
print("\n--- SANITY CHECK ---")
missing = []
for col in [TEXT_COL, LABEL_COL]:
    if col not in clf_splits["train"].column_names:
        missing.append(f"classifier missing: {col}")
for col in [SRC_COL, TGT_COL]:
    if col not in rew_splits["train"].column_names:
        missing.append(f"rewriter missing: {col}")

if missing:
    print("⚠️ Problemi colonne:")
    for m in missing:
        print(" -", m)
else:
    print("✅ Tutte le colonne richieste sono presenti.")

# ==============================
# OUTPUT: oggetti pronti per metriche
# ==============================
# -> clf_splits["test"] per metriche classificatore
# -> rew_splits["test"] per metriche riscrittore


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

=== CLASSIFIER DATASET ===
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 7968
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1992
    })
})
CLF columns: ['text', 'label']
CLF row0: {'text': 'Accents are a feature of speech and do not reflect intelligence.', 'label': 'inclusive'}


Generating data split: 0 examples [00:00, ? examples/s]


=== REWRITER DATASET ===
Rows: 4980
Cols: ['non_inclusiva', 'inclusiva']
Row0: {'non_inclusiva': 'Blonde women are usually not very smart.', 'inclusiva': "Hair color does not determine a person's intelligence."}

REWRITER SPLITS: DatasetDict({
    train: Dataset({
        features: ['non_inclusiva', 'inclusiva'],
        num_rows: 4482
    })
    test: Dataset({
        features: ['non_inclusiva', 'inclusiva'],
        num_rows: 498
    })
})

--- SANITY CHECK ---
⚠️ Problemi colonne:
 - classifier missing: label_id


Metriche Classificatore + Riscrittore

In [5]:
!pip -q install -U transformers datasets evaluate rouge_score sacrebleu bert-score scikit-learn sentencepiece accelerate

import os, json
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)
import evaluate

# ============================
# DEVICE
# ============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ============================
# FUNZIONE: trova cartella modello HF locale (dove c'è config.json)
# ============================
def find_hf_model_dir(root_dir: str) -> str:
    root_dir = os.path.abspath(root_dir)
    # 1) se root ha config.json
    if os.path.isfile(os.path.join(root_dir, "config.json")):
        return root_dir
    # 2) cerca 1 livello sotto
    for name in os.listdir(root_dir):
        p = os.path.join(root_dir, name)
        if os.path.isdir(p) and os.path.isfile(os.path.join(p, "config.json")):
            return p
    # 3) cerca ricorsivo (max 3 livelli)
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if "config.json" in filenames:
            return dirpath
    raise FileNotFoundError(f"Non trovo config.json sotto: {root_dir}")

# ============================
# PATH MODELLI (root) -> auto-detect
# ============================
CLF_ROOT = "/content/models/classifier_en"
RWR_ROOT = "/content/models/rewriter_en"

CLF_DIR = find_hf_model_dir(CLF_ROOT)
print("✅ CLF_DIR trovato:", CLF_DIR, "| files:", os.listdir(CLF_DIR)[:12])

# Il rewriter deve essere estratto correttamente; se non c'è config.json qui, fallirà con messaggio chiaro
RWR_DIR = find_hf_model_dir(RWR_ROOT)
print("✅ RWR_DIR trovato:", RWR_DIR, "| files:", os.listdir(RWR_DIR)[:12])

# ============================
# ASSUNZIONE: dataset già caricati prima
# - clf_splits (train/test) con colonne ['text','label']
# - rew_splits (train/test) con colonne ['non_inclusiva','inclusiva']
# ============================
TEXT_COL  = "text"
LABEL_COL = "label"
SRC_COL   = "non_inclusiva"
TGT_COL   = "inclusiva"

PREFIX = "Rewrite the sentence using inclusive language: "

# ============================
# 1) CLASSIFICATORE — METRICHE
# ============================
print("\n====================")
print("CLASSIFICATORE (EN)")
print("====================")

test_clf = clf_splits["test"]

# mapping label string -> id
unique_labels = sorted(list(set(test_clf[LABEL_COL])))
label2id = {lab: i for i, lab in enumerate(unique_labels)}
id2label = {i: lab for lab, i in label2id.items()}
print("Label mapping:", label2id)

clf_tokenizer = AutoTokenizer.from_pretrained(CLF_DIR, local_files_only=True)
clf_model = AutoModelForSequenceClassification.from_pretrained(CLF_DIR, local_files_only=True).to(device)
clf_model.eval()

def clf_collate(batch):
    texts = [str(x[TEXT_COL]) for x in batch]
    labels = [label2id[str(x[LABEL_COL])] for x in batch]
    enc = clf_tokenizer(texts, truncation=True, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

clf_loader = DataLoader(test_clf, batch_size=64, shuffle=False, collate_fn=clf_collate)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch in clf_loader:
        labels = batch.pop("labels").cpu().numpy()
        batch = {k: v.to(device) for k, v in batch.items()}
        out = clf_model(**batch)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)
        all_labels.extend(labels.tolist())
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
p_m, r_m, f1_m, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)

print("Accuracy:", acc)
print("Weighted  P/R/F1:", p_w, r_w, f1_w)
print("Macro     P/R/F1:", p_m, r_m, f1_m)

print("\nConfusion matrix:\n", confusion_matrix(all_labels, all_preds))
target_names = [id2label[i] for i in range(len(id2label))]
print("\nClassification report:\n", classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

if len(id2label) == 2 and len(set(all_labels.tolist())) == 2:
    try:
        roc_auc = roc_auc_score(all_labels, all_probs[:, 1])
        pr_auc  = average_precision_score(all_labels, all_probs[:, 1])
        print("ROC-AUC:", float(roc_auc))
        print("PR-AUC :", float(pr_auc))
    except Exception as e:
        print("AUC non calcolabile:", e)

# ============================
# 2) RISCRITTORE — METRICHE
# ============================
print("\n====================")
print("RISCRITTORE (EN)")
print("====================")

test_rwr = rew_splits["test"]
src_texts = [str(x) for x in test_rwr[SRC_COL]]
ref_texts = [str(x) for x in test_rwr[TGT_COL]]

rwr_tokenizer = AutoTokenizer.from_pretrained(RWR_DIR, local_files_only=True)
rwr_model = AutoModelForSeq2SeqLM.from_pretrained(RWR_DIR, local_files_only=True).to(device)
rwr_model.eval()

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
bertscore = evaluate.load("bertscore")

def batched_generate(texts, batch_size=16, max_new_tokens=64, num_beams=4):
    preds = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        inputs = [PREFIX + t for t in chunk] if PREFIX else chunk
        enc = rwr_tokenizer(inputs, truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            gen = rwr_model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                no_repeat_ngram_size=3,
                repetition_penalty=1.15,
                length_penalty=1.0,
                do_sample=False
            )
        out = rwr_tokenizer.batch_decode(gen, skip_special_tokens=True)
        preds.extend([o.strip() for o in out])
    return preds

pred_texts = batched_generate(src_texts, batch_size=16, max_new_tokens=64, num_beams=4)

rouge_res = rouge.compute(predictions=pred_texts, references=ref_texts, use_stemmer=True)
rouge_res = {k: float(v) * 100 for k, v in rouge_res.items()}
print("\nROUGE (EN):", rouge_res)

bleu_res = bleu.compute(predictions=pred_texts, references=[[r] for r in ref_texts])
print("BLEU (SacreBLEU, EN):", bleu_res)

BERTSCORE_MODEL = "roberta-base"
try:
    bs = bertscore.compute(
        predictions=pred_texts,
        references=ref_texts,
        lang="en",
        model_type=BERTSCORE_MODEL,
        batch_size=8,
        device=device
    )
except TypeError:
    bs = bertscore.compute(
        predictions=pred_texts,
        references=ref_texts,
        lang="en",
        model_type=BERTSCORE_MODEL,
        batch_size=8
    )

print("BERTScore (EN) model:", BERTSCORE_MODEL)
print("BERTScore mean P/R/F1:",
      float(np.mean(bs["precision"])),
      float(np.mean(bs["recall"])),
      float(np.mean(bs["f1"])) )

print("\n--- ESEMPI (EN) ---")
for i in range(3):
    print("\nIN  :", src_texts[i])
    print("PRED:", pred_texts[i])
    print("REF :", ref_texts[i])


Device: cuda
GPU: Tesla T4
✅ CLF_DIR trovato: /content/models/classifier_en | files: ['vocab.txt', 'special_tokens_map.json', 'model.safetensors', 'label_map.json', 'config.json', 'tokenizer.json', 'training_args.bin', 'tokenizer_config.json']
✅ RWR_DIR trovato: /content/models/rewriter_en | files: ['special_tokens_map.json', 'model.safetensors', 'generation_config.json', 'spiece.model', 'config.json', 'tokenizer.json', 'training_args.bin', 'tokenizer_config.json']

CLASSIFICATORE (EN)
Label mapping: {'inclusive': 0, 'not_inclusive': 1}
Accuracy: 0.9969879518072289
Weighted  P/R/F1: 0.996989955766664 0.9969879518072289 0.9969879487709359
Macro     P/R/F1: 0.9969899557666642 0.9969879518072289 0.9969879487709359

Confusion matrix:
 [[992   4]
 [  2 994]]

Classification report:
                precision    recall  f1-score   support

    inclusive       1.00      1.00      1.00       996
not_inclusive       1.00      1.00      1.00       996

     accuracy                           1.00

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



ROUGE (EN): {'rouge1': 35.941379952239046, 'rouge2': 13.629557650282697, 'rougeL': 31.737449095476954, 'rougeLsum': 31.806479667665972}
BLEU (SacreBLEU, EN): {'score': 10.42248982155731, 'counts': [2316, 808, 401, 232], 'totals': [6211, 5713, 5215, 4717], 'precisions': [37.28868137175978, 14.1431822159986, 7.689357622243528, 4.918380326478694], 'bp': 0.8770258554501772, 'sys_len': 6211, 'ref_len': 7026}


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore (EN) model: roberta-base
BERTScore mean P/R/F1: 0.9013653382000674 0.896730202627948 0.8989727993806204

--- ESEMPI (EN) ---

IN  : I heard he's gay, so he must be a good dancer.
PRED: Sexual orientation does not determine a person's ability to dance.
REF : Stereotypes about skills or hobbies based on sexual orientation are inaccurate.

IN  : He thinks he's always right; typical male ego.
PRED: Emotional expression varies among individuals, regardless of gender.
REF : Confidence and assertiveness in one's opinions vary among individuals and are not exclusive to any gender.

IN  : People who speak multiple languages are naturally gifted.
PRED: Language proficiency varies among individuals, not by gender.
REF : Multilingualism is typically achieved through dedicated learning and practice, not just innate talent.
